In [1]:
import numpy as np
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import matplotlib.pyplot as plt 

In [2]:
# load dataset 
df= pd.read_csv("churn-bigml-80.csv")
df.head 

<bound method NDFrame.head of      State  Account length  Area code International plan Voice mail plan  \
0       KS             128        415                 No             Yes   
1       OH             107        415                 No             Yes   
2       NJ             137        415                 No              No   
3       OH              84        408                Yes              No   
4       OK              75        415                Yes              No   
...    ...             ...        ...                ...             ...   
2661    SC              79        415                 No              No   
2662    AZ             192        415                 No             Yes   
2663    WV              68        415                 No              No   
2664    RI              28        510                 No              No   
2665    TN              74        415                 No             Yes   

      Number vmail messages  Total day minutes  Total day

In [3]:
# ststistical summary
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
State,2666,51,WV,88,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Account length,2666.0,NaN,NaN,NaN,100.620405,39.563974,1.0,73.0,100.0,127.0,243.0
Area code,2666.0,NaN,NaN,NaN,437.43886,42.521018,408.0,408.0,415.0,510.0,510.0
International plan,2666,2,No,2396,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Voice mail plan,2666,2,No,1933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Number vmail messages,2666.0,NaN,NaN,NaN,8.021755,13.612277,0.0,0.0,0.0,19.0,50.0
Total day minutes,2666.0,NaN,NaN,NaN,179.48162,54.21035,0.0,143.4,179.95,215.9,350.8
Total day calls,2666.0,NaN,NaN,NaN,100.310203,19.988162,0.0,87.0,101.0,114.0,160.0
Total day charge,2666.0,NaN,NaN,NaN,30.512404,9.215733,0.0,24.38,30.59,36.7,59.64
Total eve minutes,2666.0,NaN,NaN,NaN,200.386159,50.951515,0.0,165.3,200.9,235.1,363.7


In [4]:
# Basic dataset information

print("Dataset shape:", df.shape)

print("\nColumn information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Dataset shape: (2666, 20)

Column information:
<class 'pandas.DataFrame'>
RangeIndex: 2666 entries, 0 to 2665
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   2666 non-null   str    
 1   Account length          2666 non-null   int64  
 2   Area code               2666 non-null   int64  
 3   International plan      2666 non-null   str    
 4   Voice mail plan         2666 non-null   str    
 5   Number vmail messages   2666 non-null   int64  
 6   Total day minutes       2666 non-null   float64
 7   Total day calls         2666 non-null   int64  
 8   Total day charge        2666 non-null   float64
 9   Total eve minutes       2666 non-null   float64
 10  Total eve calls         2666 non-null   int64  
 11  Total eve charge        2666 non-null   float64
 12  Total night minutes     2666 non-null   float64
 13  Total night calls       2666 non-null   int64  
 14  Tota

In [8]:
# seperate figures and target
X= df.drop("Churn", axis=1)
y=df["Churn"]
print("Features shape", X.shape)
print("Target shape", y.shape)


print("/nTarget distribution:")
print (y.value_counts())

print("/nTarget proportions:")
print(y.value_counts(normalize=True))


Features shape (2666, 19)
Target shape (2666,)
/nTarget distribution:
Churn
False    2278
True      388
Name: count, dtype: int64
/nTarget proportions:
Churn
False    0.854464
True     0.145536
Name: proportion, dtype: float64


In [10]:
#identify categorical andp Numerical features 
categorical_features=X.select_dtypes(include=["str", "object"]).columns.tolist()
numerical_features=X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical features:")
print("categorical_features")

print("/nNumerical features:")
print(numerical_features)

Categorical features:
categorical_features
/nNumerical features:
['Account length', 'Area code', 'Number vmail messages', 'Total day minutes', 'Total day calls', 'Total day charge', 'Total eve minutes', 'Total eve calls', 'Total eve charge', 'Total night minutes', 'Total night calls', 'Total night charge', 'Total intl minutes', 'Total intl calls', 'Total intl charge', 'Customer service calls']


In [23]:
#Numerical preprocessing
numerical_transformer=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# Categorical Preprocessing
categorical_transformer=Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

#Combine preprocessing steps 
Preprocessor=ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat",categorical_transformer, categorical_features)
    ]
)

In [24]:
#Train Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Trainning features:", X_train.shape)
print("Testing features:",X_test.shape )
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Trainning features: (2132, 19)
Testing features: (534, 19)
Training target: (2132,)
Testing target: (534,)


In [25]:
# Fit PreProcessing on Training data only

X_train_processed = Preprocessor.fit_transform(X_train)

X_test_processed = Preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (2132, 71)
Processed testing shape: (534, 71)
